HW4 Problem #5 - A8.9

In [1]:
import os
import sys

# add the directory containing the notebook to Python path
sys.path.append(os.getcwd())

import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

In [3]:
P1 = np.array([[1,0,0,0],
               [0,1,0,0],
               [0,0,1,0]], dtype=float)

P2 = np.array([[ 1,0,0,0],
               [ 0,0,1,0],
               [ 0,-1,0,10]], dtype=float)

P3 = np.array([[ 1, 1, 1,-10],
               [-1, 1, 1,  0],
               [-1,-1, 1, 10]], dtype=float)

P4 = np.array([[ 0, 1, 1, 0],
               [ 0,-1, 1, 0],
               [-1, 0, 0,10]], dtype=float)

Ps = [P1, P2, P3, P4]

ys = [
    np.array([0.98, 0.93], dtype=float),
    np.array([1.01, 1.01], dtype=float),
    np.array([0.95, 1.05], dtype=float),
    np.array([2.04, 0.00], dtype=float),
]

N = len(Ps)

def unpack_camera(P):
    A = P[0:2, 0:3]
    b = P[0:2, 3]
    c = P[2, 0:3]
    d = P[2, 3]
    return A, b, c, d

cams = [unpack_camera(P) for P in Ps]

def solve_feasible(t, solver="ECOS"):
    x = cp.Variable(3)

    cons = []
    eps = 1e-9  # forces denominator to be positive
    for (A, b, c, d), y in zip(cams, ys):
        s = c @ x + d                  # scalar affine
        cons += [s >= eps]
        cons += [cp.norm(A @ x + b - s * y, 2) <= t * s]

    obj = cp.Minimize(cp.sum_squares(x))

    prob = cp.Problem(obj, cons)
    try:
        prob.solve(solver=getattr(cp, solver), verbose=False)
    except cp.SolverError:
        return None, None, "error"

    if prob.status in ("optimal", "optimal_inaccurate"):
        return x.value, prob.value, prob.status
    return None, None, prob.status

lo = 0.0
hi = 1.0

x_hi, _, status = solve_feasible(hi, solver="ECOS")
if x_hi is None:
    for _ in range(40):
        hi *= 2.0
        x_hi, _, status = solve_feasible(hi, solver="ECOS")
        if x_hi is not None:
            break

if x_hi is None:
    hi = 1.0
    x_hi, _, status = solve_feasible(hi, solver="SCS")
    for _ in range(40):
        if x_hi is not None:
            break
        hi *= 2.0
        x_hi, _, status = solve_feasible(hi, solver="SCS")

if x_hi is None:
    raise RuntimeError("Could not find any feasible t (check data / solvers).")

best_x = x_hi
best_t = hi

#Bisection
tol = 1e-4 
for _ in range(60):  
    mid = 0.5 * (lo + hi)

    x_mid, _, status = solve_feasible(mid, solver="ECOS")
    if x_mid is None:
        x_mid, _, status = solve_feasible(mid, solver="SCS")

    if x_mid is not None:
        hi = mid
        best_t = mid
        best_x = x_mid
    else:
        lo = mid

    if hi - lo <= tol:
        break

print("Bisection done.")
print("Lower bound (infeasible):", lo)
print("Upper bound (feasible):  ", best_t)
print("Gap hi-lo:", best_t - lo)
print("x* =", best_x)

#verify 
def g_value(xv):
    vals = []
    for (A, b, c, d), y in zip(cams, ys):
        s = c @ xv + d
        vals.append(np.linalg.norm((A @ xv + b) / s - y, 2))
    return max(vals), vals

gstar, per_cam = g_value(best_x)
print("g(x*) =", gstar)
print("per camera errors =", per_cam)
print("Guarantee: g(x*) - p* <= hi-lo =", best_t - lo)
dsassaaswdww

Bisection done.
Lower bound (infeasible): 0.0494384765625
Upper bound (feasible):   0.04949951171875
Gap hi-lo: 6.103515625e-05
x* = [4.91011024 5.01367819 5.18870745]
g(x*) = 0.049502852219002885
per camera errors = [0.049502852219002885, 0.03968529510272854, 0.0495022577811075, 0.04946640131292358]
Guarantee: g(x*) - p* <= hi-lo = 6.103515625e-05
